# Beginner Quant Research Tutorial

**Learning objectives:**
- Load single-asset price data
- Compute technical features using FeaturePipeline
- Build a simple mean reversion signal with Backtrader
- Run a backtest and analyze results

This tutorial demonstrates the essential workflow for researching single-asset strategies using Backtrader.


## 1. Setup

In [6]:
import sys
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from pathlib import Path

# Add project root to path
project_root = Path('/Users/zelin/Desktop/PA Investment/Invest_strategy')
sys.path.insert(0, str(project_root))

print("Setup complete")

Setup complete


## 2. Load Single Asset Data

In [7]:
# Load IVV (S&P 500 ETF) using yfinance
import yfinance as yf

ticker = "IVV"
start_date = "2020-01-01"
end_date = "2024-12-31"

# Download OHLC data
data = yf.download(ticker, start=start_date, end=end_date, progress=False)
data = data.droplevel('Ticker', axis=1)
data = data[['open', 'high', 'low', 'close', 'volume']]

# Reset index to have date as column for Backtrader
data = data.reset_index()
data.columns = ['date', 'open', 'high', 'low', 'close', 'volume']

print(f"Loaded {len(data)} rows of OHLC data")
print(f"Date range: {data['date'].min()} to {data['date'].max()}")
data.head()

KeyError: "None of [Index(['open', 'high', 'low', 'close', 'volume'], dtype='object', name='Price')] are in the [columns]"

## 3. Compute Features with FeaturePipeline

In [ ]:
from backend.research.features import FeaturePipeline

# Compute features using FeaturePipeline
pipeline = FeaturePipeline(features=['zscore_20', 'rsi_14', 'bollinger_bands'])

# FeaturePipeline expects OHLC DataFrame with specific column names
features_df = pipeline.compute(data)

# Display computed features
feature_cols = ['close', 'zscore_20', 'rsi_14', 'bb_upper', 'bb_middle', 'bb_lower']
print("Computed features (last 10 days):")
features_df[feature_cols].tail(10)

## 4. Define Backtrader Strategy (Bollinger Bands Mean Reversion)

In [ ]:
import backtrader as bt

class BollingerMeanReversion(bt.Strategy):
    """
    Mean reversion strategy using Bollinger Bands.
    - Go LONG when price crosses below lower band (oversold)
    - Go SHORT when price crosses above upper band (overbought)
    - Exit when price returns to middle band
    """
    params = (
        ('period', 20),      # Bollinger period
        ('devfactor', 2.0),  # Standard deviations
    )
    
    def __init__(self):
        # Add Bollinger Bands indicator
        self.bb = bt.indicators.BollingerBands(
            self.data.close, 
            period=self.params.period, 
            devfactor=self.params.devfactor
        )
        
        # Track signals
        self.order = None
        
    def log(self, txt, dt=None):
        dt = dt or self.data.datetime.date(0)
        # print(f'{dt.isoformat()} {txt}')  # Uncomment for debugging
        
    def notify_order(self, order):
        if order.status in [order.Submitted, order.Accepted]:
            return
        if order.status in [order.Completed]:
            if order.isbuy():
                self.log(f'BUY EXECUTED, Price: {order.executed.price:.2f}')
            else:
                self.log(f'SELL EXECUTED, Price: {order.executed.price:.2f}')
        self.order = None
        
    def next(self):
        # Check if we have a pending order
        if self.order:
            return
        
        # Get Bollinger Band values
        upper = self.bb.lines.top[0]
        lower = self.bb.lines.bot[0]
        middle = self.bb.lines.mid[0]
        close = self.data.close[0]
        
        # Not in the market - check for entry
        if not self.position:
            # Long signal: price below lower band
            if close < lower:
                self.order = self.buy()
        else:
            # In the market - check for exit
            # Exit long when price returns to middle band
            if close > middle:
                self.order = self.sell()
            # Or if price moved too far the other way, flip to short
            elif close > upper:
                self.order = self.sell()
                self.order = self.buy()  # Will execute next bar if not filled
        
print("Strategy defined: BollingerMeanReversion")

## 5. Run Backtest with BacktestEngine

In [ ]:
from backend.backtest_engine import BacktestEngine, IBKRDataFeed

# Create engine
engine = BacktestEngine(cash=100000, commission=0.001)

# Add data feed
bt_data = data.copy()
engine.add_data(IBKRDataFeed(dataname=bt_data), name='IVV')

# Add strategy
engine.add_strategy(BollingerMeanReversion)

# Run backtest
result = engine.run_backtest()

print("=== Bollinger Bands Mean Reversion Results ===")
print(f"Initial Cash: ${result['initial_cash']:,.0f}")
print(f"Final Value: ${result['final_value']:,.0f}")
print(f"Total Return: {result['total_return']*100:.2f}%")
print(f"Sharpe Ratio: {result['sharpe_ratio']:.3f}")
print(f"Max Drawdown: {result['max_drawdown']*100:.2f}%")
print(f"Total Trades: {result['total_trades']}")

## 6. Visualize Results

In [ ]:
# Plot using BacktestEngine's built-in plotter
engine.plot()

## 7. Alternative Strategy: Z-Score Mean Reversion

In [ ]:
class ZScoreMeanReversion(bt.Strategy):
    """
    Mean reversion using z-score.
    - Go LONG when z-score < -2 (oversold)
    - Go SHORT when z-score > 2 (overbought)
    - Exit when z-score crosses zero
    """
    params = (
        ('period', 20),  # Lookback period for z-score
        ('entry_threshold', 2.0),
    )
    
    def __init__(self):
        # Calculate z-score manually
        self.sma = bt.indicators.SimpleMovingAverage(self.data.close, period=self.params.period)
        self.std = bt.indicators.StandardDeviation(self.data.close, period=self.params.period)
        
    def next(self):
        if len(self) < self.params.period:
            return
            
        zscore = (self.data.close[0] - self.sma[0]) / self.std[0]
        
        if not self.position:
            if zscore < -self.params.entry_threshold:
                self.buy()
        else:
            if zscore > 0:
                self.sell()
            elif zscore < -self.params.entry_threshold * 1.5:
                # Flip to short
                self.sell()
                self.buy()
                
print("Strategy defined: ZScoreMeanReversion")

In [ ]:
# Run Z-Score strategy
engine2 = BacktestEngine(cash=100000, commission=0.001)
engine2.add_data(IBKRDataFeed(dataname=bt_data.copy()), name='IVV')
engine2.add_strategy(ZScoreMeanReversion)
result2 = engine2.run_backtest()

print("=== Z-Score Mean Reversion Results ===")
print(f"Initial Cash: ${result2['initial_cash']:,.0f}")
print(f"Final Value: ${result2['final_value']:,.0f}")
print(f"Total Return: {result2['total_return']*100:.2f}%")
print(f"Sharpe Ratio: {result2['sharpe_ratio']:.3f}")
print(f"Max Drawdown: {result2['max_drawdown']*100:.2f}%")
print(f"Total Trades: {result2['total_trades']}")

## 8. Compare Strategies

In [ ]:
# Compare both strategies
comparison = pd.DataFrame({
    'Bollinger Bands': [
        f"{result['total_return']*100:.2f}%",
        f"{result['sharpe_ratio']:.3f}",
        f"{result['max_drawdown']*100:.2f}%"
    ],
    'Z-Score': [
        f"{result2['total_return']*100:.2f}%",
        f"{result2['sharpe_ratio']:.3f}",
        f"{result2['max_drawdown']*100:.2f}%"
    ]
}, index=['Total Return', 'Sharpe Ratio', 'Max Drawdown'])

print("=== Strategy Comparison ===")
print(comparison)

## Summary

In this tutorial, you learned:
1. **Data Loading**: How to load OHLC data using yfinance
2. **Feature Engineering**: How to use FeaturePipeline for technical indicators
3. **Backtrader Strategy**: How to define custom strategies using bt.Strategy
4. **Backtesting**: How to use BacktestEngine to run backtests
5. **Comparison**: How to compare different strategy implementations

**Next steps:**
- Try different parameters for the strategies
- Add more assets (see intermediate tutorial)
- Implement trend-following strategies
